# Copula-flow relabeling with coppuccino

This bounded smoke example uses coppuccino to train a copula flow for each source and checks that the relabeled catalog recovers two synthetic clusters. The small budgets keep a clean execution practical while still exercising flow training.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import linear_sum_assignment

from petra import CopulaFlowFit, Initialization, PosteriorChain
from petra import make_catalog_copula_flows

In [ ]:
rng = np.random.default_rng(42)
truth = np.array([[-2.0, -1.0], [2.0, 1.0]])
chain = rng.normal(loc=truth, scale=0.2, size=(48, 2, 2))
for sample in chain:
    rng.shuffle(sample, axis=0)

posterior = PosteriorChain(
    chain, num_sources=2, num_params_per_source=2
)

In [ ]:
result = make_catalog_copula_flows(
    posterior,
    max_num_sources=2,
    num_iterations=1,
    rng_seed=42,
    threshold_samples=10,
    initialization=Initialization(num_iterations=5),
    flow_fit=CopulaFlowFit(
        knots=4, flow_layers=1, max_epochs=3, max_patience=2,
    ),
    progress=False,
)

In [ ]:
recovered = np.nanmean(result.chain, axis=0)
distances = np.linalg.norm(
    recovered[:, np.newaxis, :] - truth[np.newaxis, :, :], axis=-1
)
recovered_indices, truth_indices = linear_sum_assignment(distances)
matched = np.empty_like(recovered)
matched[truth_indices] = recovered[recovered_indices]
np.testing.assert_allclose(matched, truth, atol=0.15)
matched

In [ ]:
for source in range(result.num_sources):
    plt.scatter(
        result.chain[:, source, 0],
        result.chain[:, source, 1],
        s=15,
        alpha=0.6,
        label=f"Catalog source {source}",
    )
plt.xlabel("Parameter 0")
plt.ylabel("Parameter 1")
plt.legend()
plt.show()